# Medicare Part D Spending & Utilization Analysis (2020–2024)
**Complete, consolidated Python notebook — portfolio edition**

This replaces the scattered exercise cells with a reproducible analytical workflow based on the downloaded CMS annual drug-spending CSV. It covers schema checks, missingness, aggregation controls, financial reconciliation, annual metrics, selected drug analyses, outliers, spending concentration, and Tableau-ready exports.

**Run instructions:** put this notebook inside `notebooks/` and keep the supplied CSV in `data/`. Run all cells top to bottom. Database loading is provided separately in `scripts/import_to_mysql.py`; **no login prompts are required in this notebook**.

*Source:* CMS Medicare Part D Spending by Drug (reporting years 2020–2024), June 2026 methodology and dictionary. Gross spending is not net plan liability; historical suppression and varying populations limit comparisons.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Resolve project location whether run from the repository root or notebooks/.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
CSV = PROJECT_ROOT / "data" / "DSD_PTD_RY26_P04_V10_DY24_BGM.csv"
assert CSV.exists(), f"CMS CSV not found: {CSV}"
df = pd.read_csv(CSV, low_memory=False)
print("Source shape:", df.shape)
assert df.shape == (14536, 46)
display(df.head(3))

## 1. Structural review and missingness
Missing historical observations may reflect reporting/suppression or lack of historical observations. Preserve them as missing rather than replacing them with zeros.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Data types:")
df.info()
missing = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "missing_pct": df.isna().mean().mul(100).round(2)})
display(missing)
annual_missing = pd.DataFrame([{"year": year, "missing_spending_rows": int(df[f"Tot_Spndng_{year}"].isna().sum()), "missing_pct": round(df[f"Tot_Spndng_{year}"].isna().mean()*100,2)} for year in range(2020, 2025)])
display(annual_missing)

## 2. Verify row grain and prevent double counting
The source contains an overall drug record and manufacturer-level details for the same drug. For program-level totals, select `Mftr_Name == "Overall"` before aggregating.

In [ ]:
overall = df.loc[df["Mftr_Name"].eq("Overall")].copy()
print("Overall drug records:", len(overall))
print("Duplicate overall brand/generic combinations:", overall.duplicated(["Brnd_Name", "Gnrc_Name"]).sum())
naive = df["Tot_Spndng_2024"].sum()
correct = overall["Tot_Spndng_2024"].sum()
print(f"Unfiltered 2024 spending: ${naive:,.2f}")
print(f"Overall-only 2024 spending: ${correct:,.2f}")
print(f"Double-counting factor: {naive/correct:.2f}x")
assert len(overall) == 3625
assert round(correct,2) == 288667842058.40

## 3. Annual financial and utilization trends
Use **total spending divided by total fills** for the aggregate average; do not average the drug-level averages. These annual figures are not a constant cohort because historic observations differ.

In [ ]:
annual = []
for year in range(2020, 2025):
    spending = overall[f"Tot_Spndng_{year}"].sum()
    claims = overall[f"Tot_Clms_{year}"].sum()
    annual.append({"report_year": year, "gross_spending": spending, "fills": claims,
                   "spending_per_fill": spending / claims,
                   "overall_drugs_with_spending": int(overall[f"Tot_Spndng_{year}"].notna().sum())})
annual_df = pd.DataFrame(annual)
annual_df["spending_growth_pct"] = annual_df["gross_spending"].pct_change()*100
annual_df["fills_growth_pct"] = annual_df["fills"].pct_change()*100
annual_df["spending_per_fill_growth_pct"] = annual_df["spending_per_fill"].pct_change()*100
display(annual_df.round(2))
assert round(annual_df.loc[annual_df.report_year.eq(2024), 'spending_growth_pct'].iloc[0],2) == 4.66

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(annual_df["report_year"].astype(str), annual_df["gross_spending"] / 1e9, marker="o")
ax.set_title("Medicare Part D Gross Drug Spending, 2020–2024")
ax.set_ylabel("Gross spending (USD billions)")
ax.set_xlabel("Reporting year")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Financial reconciliation and outlier checks
Validate CMS average spending per claim against independently computed total spending divided by total claims, using a one-cent tolerance. Separately describe CMS outlier flags, which do not automatically indicate erroneous records.

In [ ]:
overall["calculated_spending_per_claim_2024"] = overall["Tot_Spndng_2024"] / overall["Tot_Clms_2024"]
overall["reconciliation_difference"] = (overall["calculated_spending_per_claim_2024"] - overall["Avg_Spnd_Per_Clm_2024"]).abs()
print("Max absolute discrepancy:", overall["reconciliation_difference"].max())
print("Rows exceeding $0.01:", int((overall["reconciliation_difference"] > 0.01).sum()))
print("Outlier flag counts (including missing):")
print(overall["Outlier_Flag_2024"].value_counts(dropna=False))
assert (overall["reconciliation_difference"] > .01).sum() == 0
assert overall["Outlier_Flag_2024"].eq(1).sum() == 395

In [ ]:
flagged = overall.loc[overall["Outlier_Flag_2024"].eq(1), ["Brnd_Name", "Gnrc_Name", "Tot_Spndng_2024", "Avg_Spnd_Per_Clm_2024", "Avg_Spnd_Per_Dsg_Unt_Wghtd_2024"]]
display(flagged.nlargest(10, "Tot_Spndng_2024"))

## 5. Drug-specific case: Abilify and related products
Use brand/generic names and keep formulations distinct; the aggregate data do **not** prove patient-level substitution.

In [ ]:
abilify = overall.loc[overall["Brnd_Name"].str.casefold().eq("abilify")].copy()
selected_fields = ["Brnd_Name", "Gnrc_Name", "Tot_Spndng_2023", "Tot_Spndng_2024", "Tot_Clms_2023", "Tot_Clms_2024", "Avg_Spnd_Per_Clm_2023", "Avg_Spnd_Per_Clm_2024"]
display(abilify[selected_fields])
for field in ["Tot_Spndng", "Tot_Clms", "Avg_Spnd_Per_Clm"]:
    old = abilify.iloc[0][f"{field}_2023"]
    new = abilify.iloc[0][f"{field}_2024"]
    print(f"{field} change: {(new/old - 1)*100:+.2f}%")
related = overall.loc[overall["Gnrc_Name"].str.contains("Aripiprazole", case=False, na=False)].copy()
related["claims_absolute_change"] = related["Tot_Clms_2024"] - related["Tot_Clms_2023"]
related["claims_growth_pct"] = (related["Tot_Clms_2024"] / related["Tot_Clms_2023"] - 1)*100
display(related[["Brnd_Name", "Gnrc_Name", "Tot_Clms_2023", "Tot_Clms_2024", "claims_absolute_change", "claims_growth_pct"]].sort_values("claims_absolute_change", ascending=False).round(2))

## 6. 2024 spending concentration and utilization rankings
For consistency, use overall rows only and do not equate high spending with inappropriate care.

In [ ]:
top_spending = overall.nlargest(10, "Tot_Spndng_2024")[["Brnd_Name", "Gnrc_Name", "Tot_Spndng_2024", "Tot_Clms_2024", "Avg_Spnd_Per_Clm_2024"]].copy()
top_utilization = overall.nlargest(10, "Tot_Clms_2024")[["Brnd_Name", "Tot_Clms_2024", "Tot_Spndng_2024", "Avg_Spnd_Per_Clm_2024"]].copy()
concentration = 100*top_spending["Tot_Spndng_2024"].sum()/overall["Tot_Spndng_2024"].sum()
print(f"2024 top ten share: {concentration:.2f}%")
print("Top spending medications:")
display(top_spending)
print("Most frequently filled medications:")
display(top_utilization)
assert round(concentration, 2) == 28.51

In [ ]:
high_per_fill = overall.loc[overall["Tot_Clms_2024"] >= 1000].nlargest(10, "Avg_Spnd_Per_Clm_2024")[["Brnd_Name", "Tot_Clms_2024", "Tot_Spndng_2024", "Avg_Spnd_Per_Clm_2024"]]
display(high_per_fill)

## 7. Matched-drug spending changes and driver analysis
Compare only drugs reported in both 2023 and 2024. This does **not** fully decompose national spending changes because some historical observations are missing.

In [ ]:
matched = overall.loc[
    overall["Tot_Spndng_2023"].notna() & overall["Tot_Spndng_2024"].notna() &
    overall["Tot_Clms_2023"].gt(0) & overall["Tot_Clms_2024"].gt(0)
].copy()
matched["spending_change"] = matched["Tot_Spndng_2024"] - matched["Tot_Spndng_2023"]
matched["spending_growth_pct"] = (matched["Tot_Spndng_2024"] / matched["Tot_Spndng_2023"]-1)*100
matched["claims_growth_pct"] = (matched["Tot_Clms_2024"] / matched["Tot_Clms_2023"]-1)*100
matched["spending_per_fill_growth_pct"] = ((matched["Tot_Spndng_2024"]/matched["Tot_Clms_2024"]) /
    (matched["Tot_Spndng_2023"]/matched["Tot_Clms_2023"]) - 1)*100
biggest_changes = matched.nlargest(10, "spending_change")[["Brnd_Name", "Gnrc_Name", "spending_change", "spending_growth_pct", "claims_growth_pct", "spending_per_fill_growth_pct"]]
display(biggest_changes.round(2))

## 8. Export files for Tableau
The Tableau data contain one row per overall drug per year (18,125 rows). `NULL` historical observations remain missing. The three-drug comparison is selected from the validated matched-drug calculations, not manually recomputed.

In [ ]:
annual_parts = []
for year in range(2020,2025):
    p = overall[["Brnd_Name", "Gnrc_Name", "Mftr_Name", f"Tot_Spndng_{year}", f"Tot_Clms_{year}", f"Tot_Benes_{year}", f"Avg_Spnd_Per_Clm_{year}", f"Outlier_Flag_{year}"]].copy()
    p.columns = ["Brnd_Name", "Gnrc_Name", "Mftr_Name", "Tot_Spndng", "Tot_Clms", "Tot_Benes", "Avg_Spnd_Per_Clm", "Outlier_Flag"]
    p["report_year"] = year
    annual_parts.append(p)
tableau_df = pd.concat(annual_parts, ignore_index=True)
out = PROJECT_ROOT / "data"
out.mkdir(exist_ok=True)
tableau_df.to_csv(out / "medicare_partd_tableau.csv", index=False)
select_drugs = ["Mounjaro", "Ozempic", "Jardiance"]
driver = matched.set_index("Brnd_Name").loc[select_drugs, ["claims_growth_pct", "spending_per_fill_growth_pct"]].reset_index()
driver.columns = ["Medication", "Claims_Growth_Pct", "Spending_Per_Claim_Growth_Pct"]
driver.round(2).to_csv(out / "spending_drivers_2024.csv", index=False)
print("Tableau rows exported:", len(tableau_df))
print("Tableau file:", out / "medicare_partd_tableau.csv")
display(driver.round(2))
assert len(tableau_df) == 18125

## 9. Executive interpretation and limitations
- Gross Part D spending in 2024 was **$288.67B**, increasing **4.66%** from 2023. Fills increased **5.87%**, and aggregate spending per fill declined roughly **1.14%**.
- The top ten drugs accounted for **28.51%** of 2024 spending. Eliquis ranked first by gross spending; atorvastatin led in fills.
- Mounjaro had the largest absolute increase among matched reported drugs from 2023 to 2024; utilization increased substantially as average spending per fill declined.
- Distinguish overall drug rows from manufacturer detail. Don't infer patient switching, net plan costs, fraud, price cuts, or medical inappropriateness solely from these aggregated results.
- See `reports/Executive_Report.pdf` and `references/` for further limitations and source documentation.